# GitSkills — data extraction

Downloads the [`mvaccargiu/gitskills`](https://huggingface.co/datasets/mvaccargiu/gitskills)
dataset (MSR 2027 Mining Challenge) into the project's `data/` folder and runs a first
exploration of each table.

The dataset has four tables, published as parquet parts (~13.4 GB in total):

| Table | Rows | Parquet | Content |
|---|---|---|---|
| `artifacts` | 3,797,117 | ~6.5 GB | one row per discovered `SKILL.md` occurrence: text, front matter, commit history |
| `artifact_siblings` | 7,264,865 | ~7.0 GB | scripts / reference files stored next to a representative skill |
| `repos` | 282,200 | ~20 MB | repository metadata: stars, language, fork status, license, dates |
| `mining_runs` | 7 | ~10 KB | provenance log of each collection run |

**Download is cached.** Each table gets a `.download_complete.json` marker once all of
its parts are on disk; re-running the download cell then skips it without touching the
network. Delete the marker (or pass `force=True`) to re-fetch.

In [1]:
from __future__ import annotations

import json
import time
from pathlib import Path

import pandas as pd
import pyarrow.dataset as ds
from huggingface_hub import HfApi, snapshot_download

HF_REPO_ID = "mvaccargiu/gitskills"
HF_REPO_TYPE = "dataset"

# Works whether the kernel runs in notebooks/ or at the project root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "gitskills"  # mirrors the Hub repo layout: <RAW_DIR>/data/<table>/*.parquet

TABLES = ["artifacts", "artifact_siblings", "repos", "mining_runs"]
MARKER_NAME = ".download_complete.json"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

DATA_DIR.mkdir(parents=True, exist_ok=True)
print("project root:", PROJECT_ROOT)
print("data dir    :", DATA_DIR)

project root: /Users/sebastian/Documents/Polytechnique/maitrise/LOG6307E/GitHub/LOG6307E-MSR-2027-mining-challenge
data dir    : /Users/sebastian/Documents/Polytechnique/maitrise/LOG6307E/GitHub/LOG6307E-MSR-2027-mining-challenge/data


/opt/homebrew/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Download (cached)

In [2]:
def table_dir(table: str) -> Path:
    """Local folder holding the parquet parts of one table."""
    return RAW_DIR / "data" / table


def _marker_path(table: str) -> Path:
    return table_dir(table) / MARKER_NAME


def human_bytes(n: float) -> str:
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if n < 1024 or unit == "TB":
            return f"{n:,.1f} {unit}"
        n /= 1024


def is_downloaded(table: str) -> bool:
    """True if `table` was fully downloaded before. Offline: never hits the network.

    The marker written after a successful download lists every parquet part with its
    size; the table counts as cached only if all of those files are still on disk with
    the same size (so a half-deleted or truncated download is re-fetched).
    """
    marker = _marker_path(table)
    if not marker.exists():
        return False
    try:
        info = json.loads(marker.read_text())
    except json.JSONDecodeError:
        print(f"[{table}] unreadable marker -> will re-download")
        return False
    for entry in info.get("files", []):
        path = RAW_DIR / entry["path"]
        if not path.exists() or path.stat().st_size != entry["size"]:
            print(f"[{table}] {entry['path']} missing or changed size -> will re-download")
            return False
    return bool(info.get("files"))


def download_table(table: str, force: bool = False, max_workers: int = 4) -> Path:
    """Fetch every parquet part of one table into `data/gitskills/`, once.

    Already-downloaded tables are skipped. `huggingface_hub` also resumes partial
    files, so an interrupted download can simply be re-run.
    """
    tdir = table_dir(table)
    if not force and is_downloaded(table):
        n_files = len(list(tdir.glob("*.parquet")))
        size = sum(p.stat().st_size for p in tdir.glob("*.parquet"))
        print(f"[{table}] cached: {n_files} parquet files, {human_bytes(size)} "
              f"in {tdir.relative_to(PROJECT_ROOT)} -> skipping download")
        return tdir

    print(f"[{table}] downloading from {HF_REPO_ID} ...")
    started = time.time()
    snapshot_download(
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
        local_dir=RAW_DIR,
        allow_patterns=[f"data/{table}/*.parquet"],
        max_workers=max_workers,
    )

    parts = sorted(tdir.glob("*.parquet"))
    if not parts:
        raise RuntimeError(f"[{table}] no parquet files downloaded - check the table name")

    elapsed = time.time() - started
    total = sum(p.stat().st_size for p in parts)
    _marker_path(table).write_text(json.dumps({
        "table": table,
        "repo_id": HF_REPO_ID,
        "downloaded_at": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
        "elapsed_seconds": round(elapsed, 1),
        "total_bytes": total,
        "files": [{"path": p.relative_to(RAW_DIR).as_posix(), "size": p.stat().st_size}
                  for p in parts],
    }, indent=2))
    print(f"[{table}] done: {len(parts)} files, {human_bytes(total)} in {elapsed/60:.1f} min")
    return tdir


def download_all(tables: list[str] | None = None, force: bool = False) -> dict[str, Path]:
    """Download every table (or a subset). Safe and cheap to re-run."""
    return {t: download_table(t, force=force) for t in (tables or TABLES)}

Optional check of what is on the Hub before pulling it (one API call, no data transferred).

In [3]:
def remote_inventory() -> pd.DataFrame:
    """Parquet parts and sizes per table, as published on the Hub."""
    info = HfApi().repo_info(HF_REPO_ID, repo_type=HF_REPO_TYPE, files_metadata=True)
    rows = []
    for sibling in info.siblings:
        parts = sibling.rfilename.split("/")
        if len(parts) == 3 and parts[0] == "data" and parts[2].endswith(".parquet"):
            rows.append({"table": parts[1], "size": sibling.size or 0})
    df = pd.DataFrame(rows).groupby("table").agg(files=("size", "size"), bytes=("size", "sum"))
    df["size"] = df["bytes"].map(human_bytes)
    return df.sort_values("bytes", ascending=False)


remote_inventory()

,files,bytes,size
table,,,
artifact_siblings,45,6956981334,6.5 GB
artifacts,31,6449224347,6.0 GB
repos,1,19653117,18.7 MB
mining_runs,1,2947,2.9 KB


In [4]:
# First run: downloads ~13.4 GB (make sure there is disk space for it).
# Every later run: prints "cached" per table and does nothing.
paths = download_all()

[artifacts] downloading from mvaccargiu/gitskills ...


Fetching 31 files: 100%|██████████| 31/31 [01:17<00:00,  2.49s/it]


[artifacts] done: 31 files, 6.0 GB in 1.3 min
[artifact_siblings] downloading from mvaccargiu/gitskills ...


Fetching 45 files: 100%|██████████| 45/45 [01:28<00:00,  1.96s/it]

[artifact_siblings] done: 45 files, 6.5 GB in 1.5 min
[repos] cached: 1 parquet files, 18.7 MB in data/gitskills/data/repos -> skipping download
[mining_runs] cached: 1 parquet files, 2.9 KB in data/gitskills/data/mining_runs -> skipping download


{'artifacts': PosixPath('/Users/sebastian/Documents/Polytechnique/maitrise/LOG6307E/GitHub/LOG6307E-MSR-2027-mining-challenge/data/gitskills/data/artifacts'),
 'artifact_siblings': PosixPath('/Users/sebastian/Documents/Polytechnique/maitrise/LOG6307E/GitHub/LOG6307E-MSR-2027-mining-challenge/data/gitskills/data/artifact_siblings'),
 'repos': PosixPath('/Users/sebastian/Documents/Polytechnique/maitrise/LOG6307E/GitHub/LOG6307E-MSR-2027-mining-challenge/data/gitskills/data/repos'),
 'mining_runs': PosixPath('/Users/sebastian/Documents/Polytechnique/maitrise/LOG6307E/GitHub/LOG6307E-MSR-2027-mining-challenge/data/gitskills/data/mining_runs')}

## 2. Loading helpers

In [5]:
# Columns kept for exploration: everything except the heavy free-text fields
# (`content`, commit messages), which are what makes the dataset big.
EXPLORE_COLUMNS = {
    "artifacts": [
        "repo_full_name", "filename", "location_class", "discovered_at",
        "content_fetched", "frontmatter_valid", "name", "body_chars",
        "history_fetched", "composition_fetched", "dedup_primary",
        "first_commit_at", "last_commit_at", "commit_count",
        "sibling_count", "sibling_bytes", "has_scripts", "has_references",
        "content_sha_ok", "composition_truncated",
        "first_commit_author_type", "last_commit_author_type",
    ],
    "artifact_siblings": [
        "repo_full_name", "artifact_path", "entry_name", "entry_type",
        "entry_size", "content_fetched", "skipped_reason",
    ],
    "repos": None,       # small table: load every column
    "mining_runs": None,
}

# Tables larger than this are explored on their first SAMPLE_ROWS rows, to keep
# the notebook usable on a laptop. Exact row counts are still reported.
FULL_LOAD_ROWS = 1_000_000
SAMPLE_ROWS = 250_000


def arrow_dataset(table: str) -> ds.Dataset:
    return ds.dataset(table_dir(table), format="parquet")


def row_count(table: str) -> int:
    """Exact number of rows, read from the parquet footers (no data loaded)."""
    return arrow_dataset(table).count_rows()


def disk_usage(table: str) -> int:
    return sum(p.stat().st_size for p in table_dir(table).glob("*.parquet"))


def load_table(table: str, columns="explore", n_rows="auto") -> pd.DataFrame:
    """Load a table into a pandas DataFrame.

    columns : "explore" for the light subset above, None for all columns, or a list.
    n_rows  : "auto" (full table under FULL_LOAD_ROWS rows, else the first
              SAMPLE_ROWS rows), None for everything, or an int.
    """
    dataset = arrow_dataset(table)
    cols = EXPLORE_COLUMNS.get(table) if columns == "explore" else columns
    if cols is not None:
        cols = [c for c in cols if c in dataset.schema.names]
    if n_rows == "auto":
        n_rows = None if row_count(table) <= FULL_LOAD_ROWS else SAMPLE_ROWS
    scanner = dataset.scanner(columns=cols)
    arrow_table = scanner.to_table() if n_rows is None else scanner.head(n_rows)
    return arrow_table.to_pandas()

## 3. Exploration

In [6]:
def explore(table: str, columns="explore", n_rows="auto", top_n: int = 10,
            max_unique: int = 30) -> pd.DataFrame:
    """Print a first look at one table and return the loaded DataFrame."""
    total_rows = row_count(table)
    parts = sorted(table_dir(table).glob("*.parquet"))
    schema = arrow_dataset(table).schema

    print("=" * 100)
    print(f"{table}  |  {len(parts)} parquet files  |  {human_bytes(disk_usage(table))} on disk")
    print(f"rows in dataset: {total_rows:,}  |  columns: {len(schema.names)}")
    print("-" * 100)
    print("schema:")
    for field in schema:
        print(f"  {field.name:<26} {field.type}")

    df = load_table(table, columns=columns, n_rows=n_rows)
    loaded_all = len(df) == total_rows
    print("-" * 100)
    print(f"loaded: {df.shape[0]:,} rows x {df.shape[1]} columns "
          f"({'whole table' if loaded_all else f'first {len(df):,} rows, not a random sample'})"
          f"  |  in memory: {human_bytes(df.memory_usage(deep=True).sum())}")

    print("-" * 100)
    print("head:")
    display(df.head())

    print("-" * 100)
    print("dtypes and missing values:")
    overview = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "missing_%": (df.isna().mean() * 100).round(2),
        "unique": df.nunique(dropna=True),
    })
    display(overview)

    numeric = df.select_dtypes("number")
    if not numeric.empty:
        print("-" * 100)
        print("numeric summary:")
        display(numeric.describe().T)

    # Low-cardinality text columns only: skip free text and all-distinct keys/timestamps.
    categorical = [c for c in df.columns
                   if df[c].dtype == object
                   and 0 < df[c].nunique(dropna=True) <= min(max_unique, len(df) - 1)]
    if categorical:
        print("-" * 100)
        print(f"value counts (columns with at most {max_unique} distinct values):")
        for col in categorical:
            counts = df[col].value_counts(dropna=False).head(top_n)
            share = (counts / len(df) * 100).round(2)
            display(pd.DataFrame({"count": counts, "share_%": share}).rename_axis(col))

    if not loaded_all:
        print(f"\nnote: statistics above cover the first {len(df):,} of {total_rows:,} rows. "
              f"Use load_table('{table}', n_rows=None) for the whole table.")
    return df

### 3.1 `artifacts` — one row per discovered `SKILL.md`

In [7]:
df_artifacts = explore("artifacts")

artifacts  |  31 parquet files  |  6.0 GB on disk
rows in dataset: 3,797,117  |  columns: 30
----------------------------------------------------------------------------------------------------
schema:
  repo_full_name             string
  path                       string
  filename                   string
  location_class             string
  file_sha                   string
  discovered_at              string
  content                    string
  content_fetched            int64
  frontmatter_valid          int64
  name                       string
  description                string
  body_chars                 int64
  history_fetched            int64
  composition_fetched        int64
  dedup_primary              int64
  first_commit_at            string
  last_commit_at             string
  commit_count               int64
  sibling_count              int64
  sibling_bytes              int64
  has_scripts                int64
  has_references             int64
  content_sha_ok 

,repo_full_name,filename,location_class,discovered_at,content_fetched,frontmatter_valid,name,body_chars,history_fetched,composition_fetched,dedup_primary,first_commit_at,last_commit_at,commit_count,sibling_count,sibling_bytes,has_scripts,has_references,content_sha_ok,composition_truncated,first_commit_author_type,last_commit_author_type
0,williamfiset/algorithms,SKILL.md,other,2026-07-09T17:26:45.768703+00:00,1,0.0,None,19.0,0,1,1,None,None,NaN,0.0,0.0,0.0,0.0,1.0,0.0,None,None
1,alirezarezvani/claude-skills,SKILL.md,skills-dir,2026-07-09T17:26:45.768703+00:00,0,NaN,None,NaN,0,0,0,None,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None
2,lvisei/web-developer-resources,skill.md,other,2026-07-09T17:26:45.768703+00:00,1,0.0,None,5.0,0,1,1,None,None,NaN,2.0,0.0,0.0,1.0,1.0,0.0,None,None
3,PhongFeng/FFramework,Skill.md,other,2026-07-09T17:26:45.768703+00:00,1,0.0,None,10.0,0,1,1,None,None,NaN,18.0,1917167.0,0.0,1.0,1.0,0.0,None,None
4,0xNyk/council-of-high-intelligence,SKILL.md,skills-dir,2026-07-09T17:26:45.768703+00:00,1,0.0,None,14.0,0,1,1,None,None,NaN,0.0,0.0,0.0,0.0,1.0,0.0,None,None


----------------------------------------------------------------------------------------------------
dtypes and missing values:


,dtype,non_null,missing_%,unique
repo_full_name,object,250000,0.00,38491
filename,object,250000,0.00,7
location_class,object,250000,0.00,3
discovered_at,object,250000,0.00,2
content_fetched,int64,250000,0.00,2
frontmatter_valid,float64,152176,39.13,2
name,object,117535,52.99,82436
body_chars,float64,152176,39.13,1091
history_fetched,int64,250000,0.00,2
composition_fetched,int64,250000,0.00,2


----------------------------------------------------------------------------------------------------
numeric summary:


,count,mean,std,min,25%,50%,75%,max
content_fetched,250000.0,0.608720,4.880378e-01,0.0,0.0,1.0,1.0,1.000000e+00
frontmatter_valid,152176.0,0.755073,4.300453e-01,0.0,1.0,1.0,1.0,1.000000e+00
body_chars,152176.0,450.708147,2.651999e+02,0.0,223.0,452.0,667.0,1.917000e+03
history_fetched,250000.0,0.162632,3.690303e-01,0.0,0.0,0.0,0.0,1.000000e+00
composition_fetched,250000.0,0.608272,4.881374e-01,0.0,0.0,1.0,1.0,1.000000e+00
dedup_primary,250000.0,0.607624,4.882807e-01,0.0,0.0,1.0,1.0,1.000000e+00
commit_count,40658.0,1.462664,5.161141e+00,1.0,1.0,1.0,1.0,7.080000e+02
sibling_count,152068.0,3.563939,1.558104e+02,0.0,0.0,0.0,1.0,3.855200e+04
sibling_bytes,152068.0,110960.495134,1.036532e+07,0.0,0.0,0.0,264.0,3.742128e+09
has_scripts,152068.0,0.080451,2.719908e-01,0.0,0.0,0.0,0.0,1.000000e+00


... (3 outputs left out)

,count,share_%
discovered_at,,
2026-07-09T18:07:39.150068+00:00,240297,96.12
2026-07-09T17:26:45.768703+00:00,9703,3.88


,count,share_%
first_commit_author_type,,
None,209342,83.74
User,35656,14.26
,4691,1.88
Bot,304,0.12
Organization,7,0.00


,count,share_%
last_commit_author_type,,
None,209342,83.74
User,35614,14.25
,4694,1.88
Bot,343,0.14
Organization,7,0.00



note: statistics above cover the first 250,000 of 3,797,117 rows. Use load_table('artifacts', n_rows=None) for the whole table.


In [8]:
# A few domain-specific angles on the loaded rows.
print("repositories represented:", df_artifacts["repo_full_name"].nunique())
print("\ntop repositories by number of skill files:")
display(df_artifacts["repo_full_name"].value_counts().head(10))

print("\ndeduplication representatives (dedup_primary == 1):",
      f"{df_artifacts['dedup_primary'].mean():.1%} of rows")

representatives = df_artifacts[df_artifacts["dedup_primary"] == 1]
print("\nbody size (chars) of representatives:")
display(representatives["body_chars"].describe())
print("bundle scripts:", f"{representatives['has_scripts'].mean():.1%}",
      "| bundle references:", f"{representatives['has_references'].mean():.1%}")

print("\nskill files discovered per month (first_commit_at):")
first_commit = pd.to_datetime(representatives["first_commit_at"], errors="coerce", utc=True)
display(first_commit.dt.to_period("M").value_counts().sort_index())

repositories represented: 38491

top repositories by number of skill files:


repo_full_name
majiayu000/claude-skill-registry-data    10096
ssrjkk/claude-skills                      7154
David-Li0406/meta-skill-evloving          5529
javimosch/supercli                        5176
Demerzels-lab/elsamultiskillagent         3820
liferay/liferay-portal                    2687
vamseeachanta/workspace-hub               2330
gabrielmoreira/agent-skills-mirror        2164
qyb156/PoisonedSkills                     1942
dvcrn/openclaw-skills-marketplace         1540
Name: count, dtype: int64


deduplication representatives (dedup_primary == 1): 60.8% of rows

body size (chars) of representatives:


count    151906.000000
mean        451.426606
std         264.884914
min           0.000000
25%         224.000000
50%         453.000000
75%         667.000000
max        1917.000000
Name: body_chars, dtype: float64

bundle scripts: 8.1% | bundle references: 15.9%

skill files discovered per month (first_commit_at):


/var/folders/m6/935k91bs08lb6bjclxrw2gqc0000gn/T/ipykernel_5293/4282868554.py:17: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  display(first_commit.dt.to_period("M").value_counts().sort_index())


first_commit_at
2014-06       1
2017-12       1
2018-03       1
2018-05       1
2019-12       1
2020-02       1
2020-12       1
2021-06       1
2021-09       1
2021-12       1
2022-01       1
2022-06       1
2022-08       1
2022-09       1
2023-06       2
2023-07       1
2023-10       1
2024-02       1
2024-04       2
2024-07       1
2024-09       1
2024-10       1
2024-11       2
2025-01       1
2025-02       1
2025-03       4
2025-04       1
2025-05       1
2025-07       3
2025-09       6
2025-10      40
2025-11     117
2025-12     368
2026-01    1416
2026-02    3980
2026-03    7331
2026-04    9253
2026-05    8448
2026-06    7475
2026-07    2110
Freq: M, Name: count, dtype: int64

### 3.2 `artifact_siblings` — files stored next to a representative skill

In [9]:
df_siblings = explore("artifact_siblings")

artifact_siblings  |  45 parquet files  |  6.5 GB on disk
rows in dataset: 7,264,865  |  columns: 9
----------------------------------------------------------------------------------------------------
schema:
  repo_full_name             string
  artifact_path              string
  entry_name                 string
  entry_type                 string
  entry_size                 int64
  entry_sha                  string
  content                    string
  content_fetched            int64
  skipped_reason             string
----------------------------------------------------------------------------------------------------
loaded: 250,000 rows x 7 columns (first 250,000 rows, not a random sample)  |  in memory: 78.6 MB
----------------------------------------------------------------------------------------------------
head:


,repo_full_name,artifact_path,entry_name,entry_type,entry_size,content_fetched,skipped_reason
0,0-AI-UG/cate,skills/cate-theme/SKILL.md,examples,dir,0,0,None
1,0-AI-UG/cate,skills/cate-theme/SKILL.md,examples/midnight-ember.cate-theme.json,file,3292,1,None
2,0-AI-UG/cate,skills/cate-theme/SKILL.md,examples/paper-light.cate-theme.json,file,3384,1,None
3,0-AI-UG/cate,skills/cate-theme/SKILL.md,theme.schema.json,file,6398,1,None
4,0-CYBERDYNE-SYSTEMS-0/FFT_nano,skills/runtime/fft-coder-ops/SKILL.md,references,dir,0,0,None


----------------------------------------------------------------------------------------------------
dtypes and missing values:


,dtype,non_null,missing_%,unique
repo_full_name,object,250000,0.0,6800
artifact_path,object,250000,0.0,25629
entry_name,object,250000,0.0,142803
entry_type,object,250000,0.0,2
entry_size,int64,250000,0.0,29068
content_fetched,int64,250000,0.0,3
skipped_reason,object,0,100.0,0


----------------------------------------------------------------------------------------------------
numeric summary:


,count,mean,std,min,25%,50%,75%,max
entry_size,250000.0,20503.146692,396988.893681,0.0,157.0,1529.0,5224.0,66135830.0
content_fetched,250000.0,1.176620,0.684026,0.0,1.0,1.0,2.0,2.0


----------------------------------------------------------------------------------------------------
value counts (columns with at most 30 distinct values):


,count,share_%
entry_type,,
file,197029,78.81
dir,52971,21.19



note: statistics above cover the first 250,000 of 7,264,865 rows. Use load_table('artifact_siblings', n_rows=None) for the whole table.


In [10]:
print("most common sibling file names:")
display(df_siblings["entry_name"].value_counts().head(15))

print("\nfile extensions:")
extensions = (df_siblings.loc[df_siblings["entry_type"] == "file", "entry_name"]
              .str.extract(r"(\.[^.\/]+)$", expand=False).fillna("(none)"))
display(extensions.value_counts().head(15))

print("\nsize of sibling files (bytes):")
display(df_siblings.loc[df_siblings["entry_type"] == "file", "entry_size"].describe())

print("\nsiblings per skill folder:")
display(df_siblings.groupby(["repo_full_name", "artifact_path"]).size().describe())

most common sibling file names:


entry_name
references                              11419
scripts                                  6338
agents                                   4360
agents/openai.yaml                       4214
README.md                                2332
assets                                   2259
index.json                               1219
examples                                 1140
LICENSE                                  1137
templates                                 968
references/quality-checklist.md           733
.gitignore                                728
references/api-reference.md               722
references/usage-routing.md               692
references/senior-master-standard.md      692
Name: count, dtype: int64


file extensions:


entry_name
.md       99166
.py       20982
.json     13061
.ts        8013
.yaml      7203
.js        4070
.html      4003
.sh        3709
.png       3326
.tsx       2917
(none)     2699
.txt       1953
.svg       1798
.jsx       1551
.mjs       1413
Name: count, dtype: int64


size of sibling files (bytes):


count    1.970290e+05
mean     2.601539e+04
std      4.470208e+05
min      0.000000e+00
25%      9.200000e+02
50%      2.576000e+03
75%      6.891000e+03
max      6.613583e+07
Name: entry_size, dtype: float64


siblings per skill folder:


count    27719.000000
mean         9.019084
std         32.446437
min          1.000000
25%          2.000000
50%          3.000000
75%          7.000000
max        500.000000
dtype: float64

### 3.3 `repos` — repository metadata

In [11]:
df_repos = explore("repos")

repos  |  1 parquet files  |  18.7 MB on disk
rows in dataset: 282,200  |  columns: 11
----------------------------------------------------------------------------------------------------
schema:
  full_name                  string
  owner                      string
  stars                      int64
  forks                      int64
  is_fork                    int64
  language                   string
  license                    string
  description                string
  created_at                 string
  pushed_at                  string
  metadata_fetched           int64
----------------------------------------------------------------------------------------------------
loaded: 282,200 rows x 11 columns (whole table)  |  in memory: 139.9 MB
----------------------------------------------------------------------------------------------------
head:


,full_name,owner,stars,forks,is_fork,language,license,description,created_at,pushed_at,metadata_fetched
0,williamfiset/algorithms,williamfiset,18673.0,4516.0,0.0,Java,MIT,A collection of algorithms and data structures,2017-03-18T02:50:46Z,2026-06-23T09:33:06Z,1
1,alirezarezvani/claude-skills,alirezarezvani,22655.0,3150.0,0.0,Python,MIT,345 Claude Code skills & agent skills & plugin...,2025-10-19T04:04:05Z,2026-07-14T07:46:55Z,1
2,lvisei/web-developer-resources,lvisei,46.0,11.0,0.0,,MIT,A list of web full stack resources and summarize.,2018-05-31T02:05:06Z,2023-02-09T16:00:33Z,1
3,PhongFeng/FFramework,PhongFeng,97.0,29.0,0.0,C#,,Unity + AI (Harness)解决Agent可靠性与可持续性，GameFramew...,2022-08-11T07:39:41Z,2026-07-07T06:48:02Z,1
4,0xNyk/council-of-high-intelligence,0xNyk,3564.0,319.0,0.0,Shell,MIT,Structured multi-perspective deliberation for ...,2026-03-02T08:47:09Z,2026-07-15T20:09:36Z,1


----------------------------------------------------------------------------------------------------
dtypes and missing values:


,dtype,non_null,missing_%,unique
full_name,object,282200,0.00,282200
owner,object,282200,0.00,195841
stars,float64,281685,0.18,3593
forks,float64,281685,0.18,1616
is_fork,float64,281685,0.18,2
language,object,281685,0.18,297
license,object,281685,0.18,43
description,object,169059,40.09,165475
created_at,object,281685,0.18,279425
pushed_at,object,281685,0.18,273890


----------------------------------------------------------------------------------------------------
numeric summary:


,count,mean,std,min,25%,50%,75%,max
stars,281685.0,122.150803,2405.834728,0.0,0.0,0.0,2.0,383040.0
forks,281685.0,18.623913,444.172104,0.0,0.0,0.0,0.0,80428.0
is_fork,281685.0,0.000011,0.003263,0.0,0.0,0.0,0.0,1.0
metadata_fetched,282200.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0


In [12]:
print("top languages:")
display(df_repos["language"].value_counts(dropna=False).head(15))

print("\nlicenses:")
display(df_repos["license"].value_counts(dropna=False).head(15))

print("\nstars:")
display(df_repos["stars"].describe(percentiles=[0.5, 0.9, 0.99]))

print("\nmost starred repositories with skill files:")
display(df_repos.nlargest(10, "stars")[["full_name", "stars", "forks", "language", "created_at"]])

print("\nrepositories created per month:")
created = pd.to_datetime(df_repos["created_at"], errors="coerce", utc=True)
display(created.dt.to_period("M").value_counts().sort_index().tail(24))

top languages:


language
Python        71801
TypeScript    67104
              29601
JavaScript    23938
Shell         19780
HTML          12187
Rust           8371
Go             8094
C#             5168
Java           3172
PHP            3115
PowerShell     2136
Dart           2047
C++            2022
Swift          1947
Name: count, dtype: int64


licenses:


license
                156466
MIT              85309
Apache-2.0       16889
NOASSERTION      12239
GPL-3.0           3634
AGPL-3.0          3093
BSD-3-Clause       808
None               515
Unlicense          433
GPL-2.0            407
MPL-2.0            393
CC0-1.0            333
MIT-0              278
ISC                232
CC-BY-4.0          210
Name: count, dtype: int64


stars:


count    281685.000000
mean        122.150803
std        2405.834728
min           0.000000
50%           0.000000
90%          15.000000
99%        1278.000000
max      383040.000000
Name: stars, dtype: float64


most starred repositories with skill files:


,full_name,stars,forks,language,created_at
32823,openclaw/openclaw,383040.0,80428.0,TypeScript,2025-11-24T10:16:47Z
124267,obra/superpowers,255386.0,22831.0,Shell,2025-10-09T19:45:18Z
24046,react/react,246506.0,51228.0,JavaScript,2013-05-24T16:15:54Z
9205,affaan-m/ECC,230097.0,35217.0,JavaScript,2026-01-18T00:51:51Z
249331,NousResearch/hermes-agent,218025.0,41172.0,Python,2025-07-22T22:22:28Z
38237,n8n-io/n8n,196579.0,59339.0,TypeScript,2019-06-22T09:24:21Z
28284,microsoft/vscode,187621.0,41318.0,TypeScript,2015-09-03T20:23:38Z
109822,Significant-Gravitas/AutoGPT,185565.0,46080.0,Python,2023-03-16T09:21:07Z
21179,flutter/flutter,177909.0,30815.0,Dart,2015-03-06T22:54:58Z
3019,mattpocock/skills,172208.0,14786.0,Shell,2026-02-03T11:15:53Z



repositories created per month:


/var/folders/m6/935k91bs08lb6bjclxrw2gqc0000gn/T/ipykernel_5293/2460486549.py:15: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  display(created.dt.to_period("M").value_counts().sort_index().tail(24))


created_at
2024-08      703
2024-09      688
2024-10      721
2024-11      799
2024-12      804
2025-01      910
2025-02      932
2025-03     1255
2025-04     1292
2025-05     1682
2025-06     1877
2025-07     1862
2025-08     2112
2025-09     2117
2025-10     3646
2025-11     4855
2025-12     7314
2026-01    16961
2026-02    28264
2026-03    42796
2026-04    48669
2026-05    39374
2026-06    34982
2026-07    15862
Freq: M, Name: count, dtype: int64

### 3.4 `mining_runs` — provenance of the collection

In [13]:
df_mining_runs = explore("mining_runs")

mining_runs  |  1 parquet files  |  2.9 KB on disk
rows in dataset: 7  |  columns: 7
----------------------------------------------------------------------------------------------------
schema:
  run_id                     int64
  artifact_type              string
  query                      string
  started_at                 string
  finished_at                string
  discovered                 int64
  note                       string
----------------------------------------------------------------------------------------------------
loaded: 7 rows x 7 columns (whole table)  |  in memory: 2.7 KB
----------------------------------------------------------------------------------------------------
head:


,run_id,artifact_type,query,started_at,finished_at,discovered,note
0,1,agent-skills,filename:SKILL.md,2026-07-09T17:26:45.768703+00:00,2026-07-09T18:04:34.751737+00:00,11796,None
1,2,agent-skills,filename:SKILL.md,2026-07-09T18:07:39.150068+00:00,2026-07-12T06:54:51.308396+00:00,1328473,None
2,3,agent-skills,filename:SKILL.md,2026-07-12T06:59:13.601456+00:00,2026-07-20T22:26:56.553671+00:00,0,closed retroactively: process crashed before f...
3,4,agent-skills,filename:SKILL.md,2026-07-13T12:52:52.309967+00:00,2026-07-20T22:26:56.553671+00:00,0,closed retroactively: process crashed before f...
4,5,agent-skills,filename:SKILL.md,2026-07-14T18:04:23.050325+00:00,2026-07-20T05:23:31.638481+00:00,1657948,None


----------------------------------------------------------------------------------------------------
dtypes and missing values:


,dtype,non_null,missing_%,unique
run_id,int64,7,0.00,7
artifact_type,object,7,0.00,1
query,object,7,0.00,1
started_at,object,7,0.00,7
finished_at,object,7,0.00,4
discovered,int64,7,0.00,4
note,object,4,42.86,1


----------------------------------------------------------------------------------------------------
numeric summary:


,count,mean,std,min,25%,50%,75%,max
run_id,7.0,4.000000,2.160247,1.0,2.5,4.0,5.5,7.0
discovered,7.0,428316.714286,733664.861488,0.0,0.0,0.0,670134.5,1657948.0


----------------------------------------------------------------------------------------------------
value counts (columns with at most 30 distinct values):


,count,share_%
artifact_type,,
agent-skills,7,100.0


,count,share_%
query,,
filename:SKILL.md,7,100.0


,count,share_%
finished_at,,
2026-07-20T22:26:56.553671+00:00,4,57.14
2026-07-09T18:04:34.751737+00:00,1,14.29
2026-07-12T06:54:51.308396+00:00,1,14.29
2026-07-20T05:23:31.638481+00:00,1,14.29


,count,share_%
note,,
closed retroactively: process crashed before finishing (repair-runs),4,57.14
None,3,42.86


In [14]:
display(df_mining_runs)
print("files discovered in total:", f"{df_mining_runs['discovered'].sum():,}")

,run_id,artifact_type,query,started_at,finished_at,discovered,note
0,1,agent-skills,filename:SKILL.md,2026-07-09T17:26:45.768703+00:00,2026-07-09T18:04:34.751737+00:00,11796,None
1,2,agent-skills,filename:SKILL.md,2026-07-09T18:07:39.150068+00:00,2026-07-12T06:54:51.308396+00:00,1328473,None
2,3,agent-skills,filename:SKILL.md,2026-07-12T06:59:13.601456+00:00,2026-07-20T22:26:56.553671+00:00,0,closed retroactively: process crashed before f...
3,4,agent-skills,filename:SKILL.md,2026-07-13T12:52:52.309967+00:00,2026-07-20T22:26:56.553671+00:00,0,closed retroactively: process crashed before f...
4,5,agent-skills,filename:SKILL.md,2026-07-14T18:04:23.050325+00:00,2026-07-20T05:23:31.638481+00:00,1657948,None
5,6,agent-skills,filename:SKILL.md,2026-07-20T05:28:44.667482+00:00,2026-07-20T22:26:56.553671+00:00,0,closed retroactively: process crashed before f...
6,7,agent-skills,filename:SKILL.md,2026-07-20T07:38:28.581854+00:00,2026-07-20T22:26:56.553671+00:00,0,closed retroactively: process crashed before f...


files discovered in total: 2,998,217


### 3.5 Overview of the four tables

In [15]:
dataframes = {
    "artifacts": df_artifacts,
    "artifact_siblings": df_siblings,
    "repos": df_repos,
    "mining_runs": df_mining_runs,
}

summary = pd.DataFrame([
    {
        "table": name,
        "rows_in_dataset": row_count(name),
        "rows_loaded": len(df),
        "columns_loaded": df.shape[1],
        "parquet_files": len(list(table_dir(name).glob("*.parquet"))),
        "on_disk": human_bytes(disk_usage(name)),
    }
    for name, df in dataframes.items()
])
summary

,table,rows_in_dataset,rows_loaded,columns_loaded,parquet_files,on_disk
0,artifacts,3797117,250000,22,31,6.0 GB
1,artifact_siblings,7264865,250000,7,45,6.5 GB
2,repos,282200,282200,11,1,18.7 MB
3,mining_runs,7,7,7,1,2.9 KB


The full text columns were left out of the loaded frames to keep memory down. Read them
on demand, for example:

```python
content = load_table("artifacts", columns=["repo_full_name", "path", "content"], n_rows=1000)
```